# Graph Creation

- we zip the txt files using the gzip library to compress and save precious disk space
- we offer an int_optimize flag for the edge list to optionally save an integer representation (and a word mapping)
- the integer edge list is a non negotiable when it later on comes to rustworkx which is more than 20 times faaster than pure python networkx  

In [ ]:
from src.utils import LANG_DICT
from src.graph_extraction import parse_ud_conllu, save_adjacency_list, save_edge_list

for lang in LANG_DICT.keys():
    print(f"Processing language: {lang}")
    nodes, edges = parse_ud_conllu(
        lang=lang, directed=False, use_lemma=True, rm_self_loops=True, rm_punct=True
    )
    # pure word (string) representation -> both adjacency and edgle lists 
    save_adjacency_list(nodes, edges, lang=lang, directed=False)
    save_edge_list(nodes, edges, lang=lang)
    #
    save_edge_list(nodes, edges, lang=lang, int_optimize=True) 

# Summary table

In [14]:
from src.utils import LANG_DICT
from src.graph_extraction import get_network_summary

out = f"{'Language':<8} {'N':>8} {'E':>8} {'⟨k⟩':>8} {'δ':>8}\n{'_'*48}\n"

for lang in LANG_DICT.keys():
    N, E, k, delta = get_network_summary(lang)
    out += f"{LANG_DICT[lang]:<10} {N:>8} {E:>8} {k:>8.4f} {delta:>10.6f}\n"

print(out)

Language        N        E      ⟨k⟩        δ
________________________________________________
english        4653    15473   6.6508   0.001430
arabic         4774    14963   6.2685   0.001313
czech          5302    13387   5.0498   0.000953
german         5372    15424   5.7424   0.001069
spanish        4512    16042   7.1108   0.001576
finnish        4941    11404   4.6161   0.000934
french         4617    16612   7.1960   0.001559
galician       4463    16044   7.1898   0.001611
hindi          4402    14330   6.5107   0.001479
indonesian     3708    13642   7.3581   0.001985
icelandic      4823    13540   5.6148   0.001164
italian        4802    16672   6.9438   0.001446
japanese       4894    18033   7.3694   0.001506
korean         2607     3474   2.6651   0.001023
polish         5019    13149   5.2397   0.001044
portuguese     3796    11761   6.1965   0.001633
russian        5137    13573   5.2844   0.001029
swedish        4995    14470   5.7938   0.001160
thai           4023    1

# Calculating Closeness

This cell is supposed to showcase the immense speed gain by using rustworkx. It excels due to its implementation in Rust.

In [15]:
from src.graph_extraction import load_edges
from src.closeness import nx_compute_closeness_centrality, rx_compute_closeness_centrality

edges = load_edges(lang="en", int_optimize=True)
nx_closeness = nx_compute_closeness_centrality(edges, benchmark=True)
rx_closeness = rx_compute_closeness_centrality(edges, benchmark=True)

def print_result(closeness_dict, library_name): 
    print(f"\n{library_name} closeness centrality results:")
    for i, (node, centrality) in enumerate(
        sorted(closeness_dict.items(), key=lambda x: x[0])
    ):
        if i >= 10:
            break
        print(f"Node: {node}, Closeness Centrality: {centrality}")

print_result(nx_closeness, "NetworkX")
print_result(rx_closeness, "Rustworkx")

NetworkX closeness centrality computed in 0:00:22.721058
Rustworkx closeness centrality computed in 0:00:00.491030

NetworkX closeness centrality results:
Node: 0, Closeness Centrality: 0.35680319067341615
Node: 1, Closeness Centrality: 0.33021010789324245
Node: 2, Closeness Centrality: 0.17250713835428486
Node: 3, Closeness Centrality: 0.32218297666043355
Node: 4, Closeness Centrality: 0.2594968483293356
Node: 5, Closeness Centrality: 0.18577532846132344
Node: 6, Closeness Centrality: 0.295458875833598
Node: 7, Closeness Centrality: 0.2478687127024723
Node: 8, Closeness Centrality: 0.2478687127024723
Node: 9, Closeness Centrality: 0.2478687127024723

Rustworkx closeness centrality results:
Node: 0, Closeness Centrality: 0.35680319067341615
Node: 1, Closeness Centrality: 0.33021010789324245
Node: 2, Closeness Centrality: 0.17250713835428486
Node: 3, Closeness Centrality: 0.32218297666043355
Node: 4, Closeness Centrality: 0.2594968483293356
Node: 5, Closeness Centrality: 0.1857753284613

# P Values

helpers

In [4]:
from collections import defaultdict
from src.graph_extraction import load_edges, get_network_summary
from src.monte_carlo import simulate_closeness_significance
from src.utils import LANG_DICT

lang_graphs = {}
for lang in LANG_DICT.keys():
    edges = load_edges(lang=lang, int_optimize=True)
    N, E, _, _ = get_network_summary(lang)
    lang_graphs[lang] = (edges, N, E)


def simulate(edges,N,E,T,Q=None, nullmodel=None, closeness_fn="dijkstra_all_pairs", benchmark=False):
    result = defaultdict(str)

    nullmodels = ["ER", "ES"] if nullmodel is None else [nullmodel]
    for null_model in nullmodels:
        if null_model == "ES" and isinstance(Q, list):
            for q in Q:
                result_key = f"ES_Q{q}"
                result[result_key] = (
                    simulate_closeness_significance(
                        edges, N, E, T=T, Q=q, null_model=null_model, closeness_fn=closeness_fn, seed=0, benchmark=benchmark
                    )
                )
            continue
        result[null_model] = (
            simulate_closeness_significance(
                edges, N, E, T=T, Q=Q, null_model=null_model, closeness_fn=closeness_fn, seed=0, benchmark=benchmark
            )
        )
    return result

def print_simulation_results(lang, results):
    first = True
    for null_model, (p_val, avg_orig_closeness, avg_null_closeness, std_null_closeness) in results.items():
        if not first:
            lang, avg_orig_closeness = '', ''
        print(
            f"{LANG_DICT[lang]:<15}{null_model:<15}{p_val:<15.6f}{avg_orig_closeness:<25.6f}{avg_null_closeness:<25.6f}{std_null_closeness:<25.6f}"
        )

header = f"{'Language':<15}{'nullmodel':<15}{'pvalue':<15}{'avg_orig_closeness':<25}{'avg_null_closeness':<25}{'std_null_closeness':<25}\n{'_'*120}"

The following cell performs a small benchmarking using english to decide on a feasable maximal T. Assuming approximal equal iterations/sec across all languages the results suggest that (on my machine) in one night (assuming 8h) we can calcualte p-values for all languages using the erdos renyi null model for a max T of: 8*60^2s*3.36it/s = 96,768it, which implies a max T per language of: 96,768/21 = 4608

In [ ]:
lang = "en"
edges, N, E = lang_graphs[lang]
T = 100

results = simulate(edges, N, E, T, nullmodel="ER", benchmark=True)
print(header)
print_simulation_results(lang, results)


100%|██████████| 100/100 [00:28<00:00,  3.50it/s]/s]

Language       nullmodel      pvalue         avg_orig_closeness       avg_null_closeness       std_null_closeness       
________________________________________________________________________________________________________________________
english        ER             0.000000       0.288868                 0.214057                 0.000208                 


In [11]:
import math

lang = "en"
edges, N, E = lang_graphs[lang]
T = 100
Q = max(20, math.ceil(math.log(E)))

results = simulate(edges, N, E, T, Q=Q, nullmodel="ES", benchmark=True)
print(header)
print_simulation_results(lang, results)

100%|██████████| 100/100 [00:42<00:00,  2.36it/s]/s]

Language       nullmodel      pvalue         avg_orig_closeness       avg_null_closeness       std_null_closeness       
________________________________________________________________________________________________________________________
english        ES             1.000000       0.288868                 0.295331                 0.001134                 


In [ ]:
print(header)
T = 100
Q = [10, 15, 20, 25, 30]
for lang in LANG_DICT.keys():
    edges, N, E = lang_graphs[lang]
    results = simulate(edges, N, E, T, Q=Q)
    print_simulation_results(lang, results)


Language       nullmodel      pvalue         avg_orig_closeness       avg_null_closeness       std_null_closeness       
________________________________________________________________________________________________________________________
english        ER             0.000000       0.302693                 0.221795                 0.000000                 
english        ES_Q5          1.000000       0.302693                 0.306395                 0.000000                 
english        ES_Q10         1.000000       0.302693                 0.309473                 0.000000                 
english        ES_Q15         1.000000       0.302693                 0.309687                 0.000000                 
english        ES_Q20         1.000000       0.302693                 0.306332                 0.000000                 
english        ES_Q25         1.000000       0.302693                 0.306986                 0.000000                 
english        ES_Q30         1.